In [1]:
import json
import os
import sys
from pathlib import Path
from typing import Any, TypeVar

import pydantic
from dotenv import load_dotenv
from icecream import ic
from openai import OpenAI
from tqdm import tqdm

from PydanticContracts import (
    BoundaryClarityJudgeResult,
    ChunkScoreJudgeResult,
    ContextualCoherenceJudgeResult,
    GeneralJudgeResult,
    HopeConceptUnityJudgeResult,
    HopeInformationPreservationJudgeResult,
    HopeSemanticIndependenceJudgeResult,
    IntrachunkCohesionJudgeResult,
    SizeComplianceJudgeResult,
    SyntheticChunkingExample,
)

ChecksT = TypeVar("ChecksT", bound=pydantic.BaseModel)
ResultT = TypeVar("ResultT", bound=pydantic.BaseModel)

load_dotenv()

### Generator

MODEL_NAME = "deepseek-v4-pro"
BASE_URL = "https://api.deepseek.com"
TEMPERATURE = 1.0
REASONING = False
REASONING_EFFORT = "medium"
MAX_TOKENS = (8192, 10000)[REASONING]
TIMEOUT_SECONDS = 240.0
PAIRS_PER_PROMPT = 3
REGENERATION_ATTEMPTS = 20
USE_JUDGE_FEEDBACK_ON_EVEN_ATTEMPTS = True

### Judge
JUDGE_MODEL_NAME = "deepseek-v4-pro"
JUDGE_BASE_URL = "https://api.deepseek.com"
JUDGE_TEMPERATURE = 0.0
JUDGE_REASONING = True
JUDGE_REASONING_EFFORT = "high"
JUDGE_MAX_TOKENS = (4096, 24000)[JUDGE_REASONING]
JUDGE_REGENERATION_ATTEMPTS = 20

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "prompts").is_dir() else cwd.parent
sys.path.insert(0, str(PROJECT_ROOT))

PROMPTS_ROOT = PROJECT_ROOT / "prompts"
OUTPUT_ROOT = PROJECT_ROOT / "data" / "generated"
SELECTED_PROMPTS = [
    (Path("general_validation.md"), GeneralJudgeResult),
    # (Path("metrics/size_compliance.md"), SizeComplianceJudgeResult),
    (Path("metrics/intrachunk_cohesion.md"), IntrachunkCohesionJudgeResult),
    (Path("metrics/contextual_coherence.md"), ContextualCoherenceJudgeResult),
    (Path("metrics/boundary_clarity.md"), BoundaryClarityJudgeResult),
    (Path("metrics/chunk_score.md"), ChunkScoreJudgeResult),
    (Path("metrics/hope_concept_unity.md"), HopeConceptUnityJudgeResult),
    (
        Path("metrics/hope_semantic_independence.md"),
        HopeSemanticIndependenceJudgeResult,
    ),
    (
        Path("metrics/hope_information_preservation.md"),
        HopeInformationPreservationJudgeResult,
    ),
]

ic(SELECTED_PROMPTS)

client = OpenAI(
    api_key=os.environ["API_KEY"],
    base_url=BASE_URL,
    timeout=TIMEOUT_SECONDS,
)

ic| SELECTED_PROMPTS: [(PosixPath('general_validation.md'),
                        <class 'PydanticContracts.GeneralJudgeResult'>),
                       (PosixPath('metrics/intrachunk_cohesion.md'),
                        <class 'PydanticContracts.IntrachunkCohesionJudgeResult'>),
                       (PosixPath('metrics/contextual_coherence.md'),
                        <class 'PydanticContracts.ContextualCoherenceJudgeResult'>),
                       (PosixPath('metrics/boundary_clarity.md'),
                        <class 'PydanticContracts.BoundaryClarityJudgeResult'>),
                       (PosixPath('metrics/chunk_score.md'),
                        <class 'PydanticContracts.ChunkScoreJudgeResult'>),
                       (PosixPath('metrics/hope_concept_unity.md'),
                        <class 'PydanticContracts.HopeConceptUnityJudgeResult'>),
                       (PosixPath('metrics/hope_semantic_independence.md'),
                        <class 'PydanticContracts

In [2]:
def save_json(data: dict[str, Any] | list[dict[str, Any]], path: Path) -> Path:
    """Save JSON objects in a human-readable UTF-8 file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(data, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    return path

In [3]:
def with_json_schema(
    prompt: str, result_model: type[pydantic.BaseModel]
) -> str:
    """Append a compact Pydantic JSON schema to a system prompt."""
    schema = json.dumps(
        result_model.model_json_schema(),
        ensure_ascii=False,
        separators=(",", ":"),
    )
    return f"{prompt.rstrip()}\n\nJSON schema ответа:\n{schema}"

In [4]:
def llm_judge(
    example: SyntheticChunkingExample,
    system_prompt: str,
    metric_prompt: str,
    result_model: type[ResultT],
) -> ResultT:
    messages = [
        {
            "role": "system",
            "content": with_json_schema(system_prompt, result_model),
        },
        {
            "role": "user",
            "content": (
                f"{metric_prompt}\n\n"
                "Проверь следующий синтетический пример:\n\n"
                f"{example.model_dump_json(indent=2)}"
            ),
        },
    ]

    for attempt in range(JUDGE_REGENERATION_ATTEMPTS):
        try:
            response = client.chat.completions.create(
                model=JUDGE_MODEL_NAME,
                messages=messages,
                temperature=JUDGE_TEMPERATURE,
                max_tokens=JUDGE_MAX_TOKENS,
                response_format={"type": "json_object"},
                extra_body={
                    "thinking": {"type": ("disabled", "enabled")[JUDGE_REASONING]}
                },
                reasoning_effort=JUDGE_REASONING_EFFORT,
            )
            content = response.choices[0].message.content
            # ic(response.choices[0].message)
            return result_model.model_validate_json(content)
        except pydantic.ValidationError:
            print("Retrying judging..")
            continue

    raise RuntimeError(
        f"Judge did not return valid {result_model.__name__} JSON after "
        f"{JUDGE_REGENERATION_ATTEMPTS} attempts"
    )

In [5]:
def generate(
    system_prompt: str,
    judge_system_prompt: str,
    user_prompt: str,
    judge_metric_prompt: str,
    judge_feedback_prompt: str,
    judge_result_model: type[ResultT],
):
    last_rejected_example = None
    last_judge_verdict = None

    for attempt in range(1, REGENERATION_ATTEMPTS + 1):
        messages = [
            {
                "role": "system",
                "content": with_json_schema(
                    system_prompt, SyntheticChunkingExample
                ),
            },
            {"role": "user", "content": user_prompt},
        ]
        if (
            USE_JUDGE_FEEDBACK_ON_EVEN_ATTEMPTS
            and attempt % 2 == 0
            and last_rejected_example is not None
            and last_judge_verdict is not None
        ):
            messages.extend(
                [
                    {
                        "role": "assistant",
                        "content": last_rejected_example.model_dump_json(indent=2),
                    },
                    {
                        "role": "user",
                        "content": (
                            f"{judge_feedback_prompt.rstrip()}\n\n"
                            "Полный verdict судьи:\n"
                            f"{last_judge_verdict.model_dump_json(indent=2)}"
                        ),
                    },
                ]
            )

        last_rejected_example = None
        last_judge_verdict = None

        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages,
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS,
            response_format={"type": "json_object"},
            extra_body={"thinking": {"type": ("disabled", "enabled")[REASONING]}},
            reasoning_effort=REASONING_EFFORT,
        )
        content = response.choices[0].message.content
        try:
            result = SyntheticChunkingExample.model_validate_json(content)

            print("Sending to judge..")

            judge_verdict = llm_judge(
                example=result,
                system_prompt=judge_system_prompt,
                metric_prompt=judge_metric_prompt,
                result_model=judge_result_model,
            )

            if not judge_verdict.valid:
                tqdm.write("Judge declined, retrying..")
                ic(judge_verdict)
                last_rejected_example = result
                last_judge_verdict = judge_verdict
                continue

            print("Judge accepted")

            return result.model_dump()
        except pydantic.ValidationError:
            tqdm.write("Retrying..")

    raise RuntimeError(
        f"Generator did not produce a judge-approved "
        f"{judge_result_model.__name__} example after "
        f"{REGENERATION_ATTEMPTS} attempts"
    )

In [6]:
system_prompt = (PROMPTS_ROOT / "system.md").read_text(encoding="utf-8")
judge_system_prompt = (PROMPTS_ROOT / "judge" / "system.md").read_text(encoding="utf-8")
judge_feedback_prompt = (PROMPTS_ROOT / "judge_feedback.md").read_text(encoding="utf-8")

for prompt_path, judge_result_model in tqdm(
    SELECTED_PROMPTS, desc="Prompts", position=0
):
    prompt_name = prompt_path.stem
    user_prompt = (PROMPTS_ROOT / prompt_path).read_text(encoding="utf-8")
    judge_metric_prompt = (PROMPTS_ROOT / "judge" / prompt_path).read_text(
        encoding="utf-8"
    )
    results = []
    output_path = ""
    for pair_number in tqdm(
        range(1, PAIRS_PER_PROMPT + 1), desc="Items", position=1, leave=False
    ):
        result = generate(
            system_prompt=system_prompt,
            judge_system_prompt=judge_system_prompt,
            user_prompt=user_prompt,
            judge_metric_prompt=judge_metric_prompt,
            judge_feedback_prompt=judge_feedback_prompt,
            judge_result_model=judge_result_model,
        )

        results.append(result)

        output_path = save_json(results, OUTPUT_ROOT / f"{prompt_name}.json")
    tqdm.write(f"Saved {output_path}")

Prompts:   0%|          | 0/8 [00:00<?, ?it/s]

Sending to judge..


Judge accepted
Sending to judge..


                                              
Prompts:   0%|          | 0/8 [03:10<?, ?it/s]      ic| judge_verdict: GeneralJudgeResult(valid=False, quality_score=1, issues=[JudgeIssue(severity='fatal', code='negative_error_count', message='Negative содержит 5 однотипных ошибок границ/группировки (разделы 1–6 разделены, последние пункты перенесены в следующий чанк), а не 2–3 минимальные контролируемые ошибки.'), JudgeIssue(severity='major', code='controlled_change_inaccurate', message='controlled_change неверно утверждает, что заголовок «1. Общие положения» отделён от «1.1», тогда как в negative они находятся в одном чанке; фактический сдвиг — перенос последних пунктов разделов в следующий чанк.')], checks=GeneralChecks(positive_chunks_logically_complete=True, positive_contextually_clear=True, negative_has_multiple_controlled_errors=True, negative_error_count_valid=False, size_quality_degraded=True, intrachunk_cohesion_degraded=True, contextual_coherence_degraded=True, boundary_clarit

Judge declined, retrying..
Sending to judge..


...

Prompts:  50%|█████     | 4/8 [39:43<41:48, 627.08s/it]

Judge accepted
Saved /home/vladg00dman/Projects/Work/SyntheticDocChunksGeneration/data/generated/boundary_clarity.json


Sending to judge..


Prompts:  50%|█████     | 4/8 [40:56<40:56, 614.15s/it]


KeyboardInterrupt: 